In [1]:
!pip install yfinance pandas

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

# --- CONFIGURATION ---
# List of Non-Finance, Non-Property LQ45 Companies (Representative List)
# Note: Tickers must end with .JK for Indonesia Stock Exchange
tickers = [
    # Consumer Goods
    "ICBP.JK", "INDF.JK", "UNVR.JK", "KLBF.JK", "AMRT.JK", "GGRM.JK", "HMSP.JK",
    # Mining & Energy
    "ADRO.JK", "PTBA.JK", "ITMG.JK", "MDKA.JK", "INCO.JK", "MEDC.JK", "PGAS.JK", "ANTM.JK",
    # Infrastructure & Telco
    "TLKM.JK", "EXCL.JK", "ISAT.JK", "JSMR.JK", "TOWR.JK",
    # Industrials & Basic Materials
    "ASII.JK", "SMGR.JK", "UNTR.JK", "INKP.JK",
    # Tech / Others
    "GOTO.JK", "EMTK.JK", "AKRA.JK"
]

def calculate_roic(ticker_symbol):
    print(f"Processing {ticker_symbol}...")
    try:
        stock = yf.Ticker(ticker_symbol)

        # specific period fetch is limited in free tier, we get max available
        financials = stock.financials
        balance_sheet = stock.balance_sheet

        if financials.empty or balance_sheet.empty:
            print(f"No data found for {ticker_symbol}")
            return None

        # Align columns (dates)
        common_dates = financials.columns.intersection(balance_sheet.columns)
        financials = financials[common_dates]
        balance_sheet = balance_sheet[common_dates]

        roic_data = {}

        for date in common_dates:
            try:
                # --- 1. Get NOPAT (Net Operating Profit After Tax) ---
                # EBIT is usually 'EBIT' or 'Operating Income'
                ebit = financials.loc['EBIT', date] if 'EBIT' in financials.index else financials.loc['Operating Income', date]

                # Calculate Effective Tax Rate
                pretax_income = financials.loc['Pretax Income', date]
                tax_provision = financials.loc['Tax Provision', date]

                if pretax_income != 0:
                    tax_rate = tax_provision / pretax_income
                else:
                    tax_rate = 0.22 # Default fallback

                # Cap tax rate between 0% and 40% to remove anomalies
                tax_rate = max(0.0, min(tax_rate, 0.40))

                nopat = ebit * (1 - tax_rate)

                # --- 2. Get Invested Capital ---
                # Formula: Total Debt + Total Equity - Cash
                # Note: yfinance fields can vary slightly

                total_equity = balance_sheet.loc['Stockholders Equity', date]

                # Try to find Total Debt
                if 'Total Debt' in balance_sheet.index:
                    total_debt = balance_sheet.loc['Total Debt', date]
                elif 'Long Term Debt' in balance_sheet.index: # Approximation if Total Debt missing
                    total_debt = balance_sheet.loc['Long Term Debt', date]
                else:
                    total_debt = 0

                # Cash
                cash = balance_sheet.loc['Cash And Cash Equivalents', date]

                invested_capital = total_debt + total_equity - cash

                # Avoid division by zero
                if invested_capital == 0:
                    roic = np.nan
                else:
                    roic = (nopat / invested_capital) * 100 # Percentage

                roic_data[date.strftime('%Y-%m-%d')] = roic

            except KeyError as e:
                # Field missing for this specific year
                continue

        # Create series for this stock
        return pd.Series(roic_data, name=ticker_symbol.replace('.JK', ''))

    except Exception as e:
        print(f"Error calculating {ticker_symbol}: {e}")
        return None

# --- MAIN EXECUTION ---
results = []
for t in tickers:
    data = calculate_roic(t)
    if data is not None:
        results.append(data)

# Combine into DataFrame
if results:
    df_roic = pd.concat(results, axis=1).T  # Transpose so Stocks are rows, Years are columns

    # Sort columns chronologically
    df_roic = df_roic.reindex(sorted(df_roic.columns), axis=1)

    # Sort by the most recent year's ROIC (High to Low)
    last_col = df_roic.columns[-1]
    df_roic = df_roic.sort_values(by=last_col, ascending=False)

    # Format output
    pd.options.display.float_format = '{:.2f}%'.format
    print("\n--- ROIC Table (Percentage) ---")
    print(df_roic)

    # Save to CSV
    df_roic.to_csv("LQ45_ROIC_Analysis.csv")
    print("\nFile saved as LQ45_ROIC_Analysis.csv")
else:
    print("No data retrieved.")

Processing ICBP.JK...
Processing INDF.JK...
Processing UNVR.JK...
Processing KLBF.JK...
Processing AMRT.JK...
Processing GGRM.JK...
Processing HMSP.JK...
Processing ADRO.JK...
Processing PTBA.JK...
Processing ITMG.JK...
Processing MDKA.JK...
Processing INCO.JK...
Processing MEDC.JK...
Processing PGAS.JK...
Processing ANTM.JK...
Processing TLKM.JK...
Processing EXCL.JK...
Processing ISAT.JK...
Processing JSMR.JK...
Processing TOWR.JK...
Processing ASII.JK...
Processing SMGR.JK...
Processing UNTR.JK...
Processing INKP.JK...
Processing GOTO.JK...
Processing EMTK.JK...
Processing AKRA.JK...

--- ROIC Table (Percentage) ---
      2021-12-31  2022-12-31  2023-12-31  2024-12-31
UNVR      87.63%     113.64%     156.94%      99.25%
ITMG      83.62%     209.10%      50.61%      37.45%
PTBA      39.22%      55.95%      34.32%      25.61%
HMSP      61.23%      24.99%      29.39%      25.29%
AMRT      24.63%      29.22%      29.50%      24.79%
UNTR      24.59%      46.98%      27.80%      23.55%
AK

In [7]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# --- CONFIGURATION ---
# Same universe (Non-Finance, Non-Property LQ45)
tickers = [
    "ICBP.JK", "INDF.JK", "UNVR.JK", "KLBF.JK", "AMRT.JK", "GGRM.JK", "HMSP.JK", # Consumer
    "ADRO.JK", "PTBA.JK", "ITMG.JK", "MDKA.JK", "INCO.JK", "MEDC.JK", "PGAS.JK", "ANTM.JK", # Energy/Mining
    "TLKM.JK", "EXCL.JK", "ISAT.JK", "JSMR.JK", "TOWR.JK", # Infra/Telco
    "ASII.JK", "SMGR.JK", "UNTR.JK", "INKP.JK", # Industrial
    "GOTO.JK", "EMTK.JK", "AKRA.JK" # Tech/Others
]

# Set Timeframe (10 Years)
end_date = datetime.now()
start_date = end_date - timedelta(days=365 * 10)

print(f"Fetching data from {start_date.date()} to {end_date.date()}...")

# Bulk download Adjusted Close prices (Factoring in Dividends & Splits)
# auto_adjust is True by default, meaning 'data' will have ticker symbols as columns with adjusted prices
data = yf.download(tickers, start=start_date, end=end_date, progress=True)

tsr_results = []

for ticker in tickers:
    try:
        # Get the series for the ticker and drop NaN values (days when market was closed or stock not listed)
        # Access the 'Close' column which now contains the adjusted prices for the specific ticker
        series = data['Close'][ticker].dropna()

        if series.empty:
            continue

        # Determine Start and End Prices
        start_price = series.iloc[0]
        end_price = series.iloc[-1]

        # Get actual start date (handles companies listed < 10 years ago)
        actual_start_date = series.index[0]
        years_active = (series.index[-1] - actual_start_date).days / 365.25

        # 1. Calculate Absolute TSR ((End - Start) / Start)
        total_return = (end_price - start_price) / start_price

        # 2. Calculate CAGR (Compound Annual Growth Rate)
        # Formula: (End/Start)^(1/n) - 1
        if years_active > 0:
            cagr = (end_price / start_price) ** (1 / years_active) - 1
        else:
            cagr = 0

        tsr_results.append({
            'Ticker': ticker.replace('.JK', ''),
            'Start Date': actual_start_date.strftime('%Y-%m-%d'),
            'Years Listed': round(years_active, 1),
            'Start Price (Adj)': round(start_price, 0),
            'Current Price (Adj)': round(end_price, 0),
            'Total Return (TSR) %': round(total_return * 100, 2),
            'Annualized Return (CAGR) %': round(cagr * 100, 2)
        })

    except Exception as e:
        print(f"Error calculating {ticker}: {e}")

# Create DataFrame
df_tsr = pd.DataFrame(tsr_results)

# Sort by Total Return (Highest to Low)
df_tsr = df_tsr.sort_values(by='Total Return (TSR) %', ascending=False)

# Formatting for display
pd.options.display.float_format = '{:,.2f}'.format

print("\n" + "="*80)
print(f"TOTAL SHAREHOLDER RETURN (TSR) ANALYSIS (Last 10 Years)")
print("Based on Adjusted Close prices (includes Dividends & Splits)")
print("="*80)
print(df_tsr[['Ticker', 'Start Date', 'Total Return (TSR) %', 'Annualized Return (CAGR) %']].to_string(index=False))

# Save full details to CSV
df_tsr.to_csv("LQ45_10Year_TSR.csv", index=False)
print("\nDetailed data saved to 'LQ45_10Year_TSR.csv'")

Fetching data from 2016-02-03 to 2026-01-31...


/tmp/ipython-input-2026126261.py:24: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start=start_date, end=end_date, progress=True)
[*********************100%***********************]  27 of 27 completed


TOTAL SHAREHOLDER RETURN (TSR) ANALYSIS (Last 10 Years)
Based on Adjusted Close prices (includes Dividends & Splits)
Ticker Start Date  Total Return (TSR) %  Annualized Return (CAGR) %
  ITMG 2016-02-03              1,609.15                       32.86
  ADRO 2016-02-03              1,499.39                       31.98
  ANTM 2016-02-03              1,426.83                       31.37
  MEDC 2016-02-03              1,284.76                       30.09
  INKP 2016-02-03                973.05                       26.81
  PTBA 2016-02-03                837.38                       25.11
  MDKA 2016-02-03                802.71                       24.64
  INCO 2016-02-03                376.57                       16.92
  UNTR 2016-02-03                222.88                       12.45
  AMRT 2016-02-03                211.44                       12.04
  ISAT 2016-02-03                113.81                        7.90
  TLKM 2016-02-03                 73.57                        5.6